## Step up cryptographic environment

In [ ]:
%pip install cryptography

In [ ]:
import os
import json
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.exceptions import InvalidTag


In [ ]:
import cryptography

print("Cryptography version:", cryptography.__version__)
print("AES-GCM ready")

Cryptography version: 50.0.2
AES-GCM ready


## Defining the encryption protocol

In [ ]:
class HealthcareProtocol:

    def __init__(self, flawed=False):

        self.key = AESGCM.generate_key(bit_length=256)
        self.aesgcm = AESGCM(self.key)

        self.counter = 0
        self.flawed = flawed

    def encrypt(self, plaintext, metadata):

        if self.counter >= 2**96:
            raise ValueError("Nonce space exhausted")

        # Reusing nonce in the flawed implementation
        nonce_number = 0 if self.flawed else self.counter

        nonce = nonce_number.to_bytes(12, "big")

        self.counter += 1

        aad = json.dumps(
            metadata, sort_keys=True
        ).encode("utf-8")

        encrypted = self.aesgcm.encrypt(
            nonce,
            plaintext.encode("utf-8"),
            aad
        )

        return {
            "nonce": nonce,
            "ciphertext": encrypted[:-16],
            "tag": encrypted[-16:],
            "aad": aad
        }

    def decrypt(self, packet):

        plaintext = self.aesgcm.decrypt(
            packet["nonce"],
            packet["ciphertext"] + packet["tag"],
            packet["aad"]
        )

        return plaintext.decode("utf-8")

### Patient metadata

In [ ]:
metadata = {
    "patient_id": "P1001",
    "sender": "Hospital_A",
    "recipient": "GP_B",
    "key_version": 1
}

### Testing secure encryption

In [ ]:
secure = HealthcareProtocol(flawed=False)

patient_record = "Patient requires follow-up appointment"

packet = secure.encrypt(patient_record, metadata)

print("Nonce:", packet["nonce"].hex())
print("Ciphertext:", packet["ciphertext"].hex())
print("Authentication Tag:", packet["tag"].hex())

decrypted = secure.decrypt(packet)

print("Decrypted Record:", decrypted)
print("Successful:", decrypted == patient_record)

Nonce: 000000000000000000000000
Ciphertext: 2edeb0fb774bed825aa2311ab5b654c1f204edb9c7cdf860a2784297768178c2d10bdbf1767f
Authentication Tag: 26a61cabf444ce3ed6fe8b17bd12511b
Decrypted Record: Patient requires follow-up appointment
Successful: True


## Security Testing
### Creating test data

In [ ]:
test_record = "Patient requires urgent medical review"

test_packet = secure.encrypt(test_record, metadata)

print("Test packet generated")

### Testing successful decryption

In [ ]:
result = secure.decrypt(test_packet)

print("Original:", test_record)
print("Decrypted:", result)
print("Test passed:", result == test_record)

### Testing modified ciphertext

In [ ]:
tampered_packet = test_packet.copy()

tampered_ciphertext = bytearray(tampered_packet["ciphertext"])
tampered_ciphertext[0] ^= 1

tampered_packet["ciphertext"] = bytes(tampered_ciphertext)

try:
    secure.decrypt(tampered_packet)
    print("Test failed: Modified ciphertext accepted")

except InvalidTag:
    print("Test passed: Modified ciphertext rejected")

### Testing modified metadata

In [ ]:
modified_packet = test_packet.copy()

modified_packet["aad"] = json.dumps({
    "patient_id": "P2002",
    "sender": "Hospital_A",
    "recipient": "GP_B",
    "key_version": 1
}, sort_keys=True).encode("utf-8")

try:
    secure.decrypt(modified_packet)
    print("Test failed: Modified metadata accepted")

except InvalidTag:
    print("Test passed: Modified metadata rejected")

### Testing incorrect key

In [ ]:
wrong_key = AESGCM.generate_key(bit_length=256)
wrong_aesgcm = AESGCM(wrong_key)

try:
    wrong_aesgcm.decrypt(
        test_packet["nonce"],
        test_packet["ciphertext"] + test_packet["tag"],
        test_packet["aad"]
    )

    print("Test failed: Incorrect key accepted")

except InvalidTag:
    print("Test passed: Incorrect key rejected")

### Testing results

In [ ]:
print("SECURITY TEST SUMMARY")
print("---------------------")
print("Successful decryption: Expected to pass")
print("Modified ciphertext: Expected rejection")
print("Modified metadata: Expected rejection")
print("Incorrect key: Expected rejection")